In [ ]:

import os, json, re, calendar as _cal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import duckdb

BASE = Path('/Users/nick/Desktop/intern/POLYMARKET')
os.chdir(BASE)

con = duckdb.connect()
con.execute("SET threads=4; SET memory_limit='3GB'; SET preserve_insertion_order=false;")

# ── Lookups ────────────────────────────────────────────────────────────────
clob = json.loads((BASE / 'all_elon_cids_2026.json').read_text())
mids = json.loads((BASE / 'market_ids.json').read_text())

cid_to_slug = {
    m['condition_id']: m['market_slug']
    for m in clob['markets']
    if m.get('condition_id') and m.get('market_slug')
    and re.match(r'elon-musk-of-tweets-', m['market_slug'])
}

def slug_to_bucket(slug):
    m = re.search(r'(\d+-\d+)$', slug);   return m.group(1) if m else (re.search(r'(\d+plus)$', slug) or type('', (), {'group': lambda s,i: None})()).group(1)

def slug_to_market(slug):
    m = re.search(r'of-tweets-(january|february|march|april|may|june|july|august|september|october|november|december)-(\d{4})-', slug)
    if m: return f"{m.group(1).capitalize()} {m.group(2)} monthly"
    m = re.search(r'of-tweets-([a-z]+-\d+)-([a-z]+-\d+)', slug)
    if m: return f"{m.group(1)} – {m.group(2)} weekly"
    return slug

def bucket_bounds(b):
    if b is None: return None, None, None
    m = re.match(r'^(\d+)-(\d+)$', str(b))
    if m: lo, hi = float(m.group(1)), float(m.group(2)); return lo, hi, (lo+hi)/2
    m = re.match(r'^(\d+)plus$', str(b))
    if m: lo = float(m.group(1)); return lo, None, None
    return None, None, None

rows = []
for cid, slug in cid_to_slug.items():
    b = slug_to_bucket(slug); mn = slug_to_market(slug)
    lo, hi, mid = bucket_bounds(b)
    if b and lo is not None:
        rows.append({'condition_id': cid, 'bucket': b, 'market_name': mn,
                     'bucket_lo': lo, 'bucket_hi': hi, 'bucket_mid': mid})
bucket_df = pd.DataFrame(rows)
for mn, grp in bucket_df.groupby('market_name'):
    widths = (grp['bucket_hi'] - grp['bucket_lo']).dropna()
    bw = widths.median() if len(widths) > 0 else 20.0
    mask = bucket_df['market_name'].eq(mn) & bucket_df['bucket_hi'].isna()
    bucket_df.loc[mask, 'bucket_hi']  = bucket_df.loc[mask, 'bucket_lo'] + bw
    bucket_df.loc[mask, 'bucket_mid'] = bucket_df.loc[mask, 'bucket_lo'] + bw / 2

cid_to_market = {r['condition_id']: r['market_name'] for _, r in bucket_df.iterrows()}
cid_to_bucket_mid = {r['condition_id']: r['bucket_mid'] for _, r in bucket_df.iterrows()}

known_rows = [{'asset_id': m.get('yes_token'), 'condition_id': m.get('contract_address',''), 'side':'YES'}
              for m in mids if m.get('yes_token')] + \
             [{'asset_id': m.get('no_token'),  'condition_id': m.get('contract_address',''), 'side':'NO'}
              for m in mids if m.get('no_token')]
known_sides_df = pd.DataFrame(known_rows)

# Market expiry dates
cid_to_expiry = {}
for m in clob['markets']:
    if m.get('condition_id') and m.get('end_date_iso'):
        try:
            cid_to_expiry[m['condition_id']] = pd.to_datetime(m['end_date_iso'], utc=True)
        except: pass

# Add expiry to bucket_df
bucket_df['expiry'] = bucket_df['condition_id'].map(cid_to_expiry)

con.execute("CREATE OR REPLACE TEMP TABLE bucket_labels AS SELECT * FROM bucket_df")
con.execute("CREATE OR REPLACE TEMP TABLE known_sides  AS SELECT * FROM known_sides_df")

ELON_CID_SQL = '(' + ', '.join(f"'{c}'" for c in cid_to_slug) + ')'
SOURCE = '1s'   # change to 'master' if elon_ticks_1s.parquet not yet built
print(f"Markets: {bucket_df['market_name'].nunique()}  |  Buckets: {len(bucket_df)}  |  Source: {SOURCE}")


In [ ]:

# ── Build ohlcv_df (same DuckDB pipeline as data_pipeline.ipynb) ───────────
src_file = str(BASE / ('elon_ticks_1s.parquet' if SOURCE == '1s' else 'elon_tweet_ticks.parquet'))

if SOURCE == '1s':
    src_q = f"""SELECT timestamp, condition_id, asset_id, best_bid, best_ask,
                       trade_size AS size,
                       CASE WHEN n_trades > 0 THEN 1 ELSE 0 END AS is_trade
                FROM read_parquet('{src_file}')
                WHERE condition_id IN {ELON_CID_SQL} AND best_bid IS NOT NULL"""
else:
    src_q = f"""SELECT timestamp, condition_id, asset_id, best_bid, best_ask, size,
                       CASE WHEN event_type = 'last_trade_price' THEN 1 ELSE 0 END AS is_trade
                FROM read_parquet('{src_file}')
                WHERE condition_id IN {ELON_CID_SQL}
                  AND event_type IN ('price_change','last_trade_price')
                  AND best_bid IS NOT NULL"""

ohlcv_df = con.execute(f"""
WITH src AS ({src_q}),
avg_m  AS (SELECT condition_id, asset_id, AVG((best_bid+best_ask)/2) AS avg_mid FROM src GROUP BY 1,2),
ranked AS (SELECT condition_id, asset_id, RANK() OVER (PARTITION BY condition_id ORDER BY avg_mid) AS rk FROM avg_m),
kyc    AS (SELECT DISTINCT condition_id FROM known_sides WHERE side='YES'),
yes_t  AS (
    SELECT asset_id, condition_id FROM known_sides WHERE side='YES'
    UNION ALL
    SELECT r.asset_id, r.condition_id FROM ranked r WHERE r.rk=1 AND r.condition_id NOT IN (SELECT condition_id FROM kyc)
),
hourly AS (
    SELECT s.condition_id, DATE_TRUNC('hour', s.timestamp) AS ts,
           arg_min((s.best_bid+s.best_ask)/2, s.timestamp) AS open,
           MAX((s.best_bid+s.best_ask)/2)  AS high,
           MIN((s.best_bid+s.best_ask)/2)  AS low,
           arg_max((s.best_bid+s.best_ask)/2, s.timestamp) AS close,
           AVG(s.best_ask - s.best_bid)    AS spread,
           SUM(s.size * s.is_trade)        AS volume
    FROM src s JOIN yes_t yt ON s.condition_id=yt.condition_id AND s.asset_id=yt.asset_id
    GROUP BY s.condition_id, DATE_TRUNC('hour', s.timestamp)
)
SELECT h.ts AS timestamp, h.condition_id, bl.bucket, bl.market_name,
       bl.bucket_lo, bl.bucket_hi, bl.bucket_mid,
       h.open, h.high, h.low, h.close, h.spread, COALESCE(h.volume,0) AS volume
FROM hourly h
JOIN bucket_labels bl ON h.condition_id = bl.condition_id
ORDER BY bl.market_name, bl.bucket_mid, h.ts
""").df()

ohlcv_df['timestamp'] = pd.to_datetime(ohlcv_df['timestamp'], utc=True)

# Add DTE (days to expiry) — join expiry date from bucket_df
expiry_map = bucket_df.drop_duplicates('market_name').set_index('market_name')['expiry']
ohlcv_df['expiry'] = ohlcv_df['market_name'].map(expiry_map)
ohlcv_df['dte']    = (ohlcv_df['expiry'] - ohlcv_df['timestamp']).dt.total_seconds() / 86400

print(f"ohlcv_df: {len(ohlcv_df):,} rows  |  {ohlcv_df['market_name'].nunique()} markets  |  {ohlcv_df['bucket'].nunique()} buckets")
print(f"Date range: {ohlcv_df['timestamp'].min().date()} → {ohlcv_df['timestamp'].max().date()}")
print(f"\nBuckets per market:")
print(ohlcv_df.groupby('market_name')['bucket'].nunique().sort_values(ascending=False).to_string())


In [ ]:

# ── Select focus market (richest bucket set) ───────────────────────────────
MARKET = ohlcv_df.groupby('market_name')['bucket'].nunique().idxmax()
mdf    = ohlcv_df[ohlcv_df['market_name'] == MARKET].copy()

print(f"Focus market: {MARKET}  ({mdf['bucket'].nunique()} buckets, {mdf['timestamp'].nunique()} hourly obs)")

# ── Interactive price chart (Plotly) ──────────────────────────────────────
# Color each bucket by its average probability — high-prob = warm, low-prob = cool.
# Labels only on hover; legend shows 6 sampled buckets to avoid clutter.
avg_prob = mdf.groupby('bucket_mid')['close'].mean().sort_index()
bucket_mids = avg_prob.index.tolist()
N = len(bucket_mids)

# Sample ~6 buckets for legend; all are visible and hoverable
legend_show = set(
    [bucket_mids[0], bucket_mids[N//5], bucket_mids[2*N//5],
     bucket_mids[3*N//5], bucket_mids[4*N//5], bucket_mids[-1]]
)

colorscale = px.colors.diverging.RdYlBu_r
def prob_to_color(p, mn=avg_prob.min(), mx=avg_prob.max()):
    idx = int((p - mn) / max(mx - mn, 1e-9) * (len(colorscale)-1))
    return colorscale[max(0, min(idx, len(colorscale)-1))]

fig = go.Figure()
for bm in sorted(bucket_mids):
    sub  = mdf[mdf['bucket_mid'] == bm].sort_values('timestamp')
    bkt  = sub['bucket'].iloc[0] if len(sub) else str(bm)
    p_avg = avg_prob[bm]
    fig.add_trace(go.Scatter(
        x=sub['timestamp'], y=sub['close'],
        mode='lines', line=dict(width=1.2, color=prob_to_color(p_avg)),
        name=f"{bkt} ({p_avg:.0%})",
        showlegend=(bm in legend_show),
        legendgroup=bkt,
        hovertemplate=(
            f"<b>Bucket {bkt}</b><br>"
            "Time: %{x|%b %d %H:%M}<br>"
            "Prob: %{y:.3f}<br>"
            f"Avg prob: {p_avg:.1%}<extra></extra>"
        )
    ))

fig.update_layout(
    title=f"<b>{MARKET}</b> — YES bucket probabilities over time<br>"
          "<sup>Colour = avg probability (red=likely, blue=unlikely). "
          "Legend shows 6 representative buckets; all buckets show on hover.</sup>",
    xaxis_title="Date", yaxis_title="Implied probability",
    yaxis=dict(tickformat='.0%', range=[0, 1]),
    height=500, hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9))
)
fig.show()


In [ ]:

# ── PMF heatmap: how the probability mass shifts over time (Plotly) ────────
price_matrix = (
    mdf.pivot_table(index='timestamp', columns='bucket_mid', values='close', aggfunc='last')
    .ffill(limit=3)
    .dropna(thresh=int(mdf['bucket_mid'].nunique() * 0.7))
)
# Normalise each row so probabilities sum to 1
price_matrix = price_matrix.div(price_matrix.sum(axis=1), axis=0)

bm_cols = sorted(price_matrix.columns)
ts_vals = price_matrix.index

# Bucket labels for y-axis (only every ~5th to avoid crowding)
step     = max(1, len(bm_cols) // 15)
ytick_vals = [bm_cols[i] for i in range(0, len(bm_cols), step)]
ytick_text = [str(mdf[mdf['bucket_mid'] == v]['bucket'].iloc[0])
              if len(mdf[mdf['bucket_mid'] == v]) else str(v) for v in ytick_vals]

fig = go.Figure(go.Heatmap(
    x=ts_vals,
    y=bm_cols,
    z=price_matrix[bm_cols].T.values,
    colorscale='Viridis',
    colorbar=dict(title='Probability', tickformat='.0%'),
    hovertemplate='Time: %{x|%b %d %H:%M}<br>Bucket mid: %{y} tweets<br>Prob: %{z:.3f}<extra></extra>',
    zmin=0, zmax=price_matrix.values.max()
))
fig.update_layout(
    title=f"<b>PMF Evolution</b> — {MARKET}<br>"
          "<sup>Colour intensity = probability mass. Watch the peak (hot spot) shift as new info arrives.</sup>",
    xaxis_title="Date",
    yaxis=dict(title="Bucket (tweet count)", tickvals=ytick_vals, ticktext=ytick_text),
    height=480
)
fig.show()


In [ ]:

# ── Trade flow: volume over time + buy/sell imbalance (Plotly) ────────────
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=['Hourly Volume by Bucket (top 8)',
                                    'Total Volume across all Markets (daily)',
                                    'Volume-weighted Average Spread'])

# Panel 1: top-8 buckets by total volume, stacked area
top8 = mdf.groupby('bucket_mid')['volume'].sum().nlargest(8).index
colors8 = px.colors.qualitative.Plotly
for k, bm in enumerate(sorted(top8)):
    sub = mdf[mdf['bucket_mid'] == bm].sort_values('timestamp')
    bkt = sub['bucket'].iloc[0]
    fig.add_trace(go.Bar(
        x=sub['timestamp'], y=sub['volume'],
        name=bkt, marker_color=colors8[k % len(colors8)],
        hovertemplate=f"<b>Bucket {bkt}</b><br>Hour: %{{x|%b %d %H:%M}}<br>Vol: $%{{y:,.0f}}<extra></extra>",
        legendgroup='vol', showlegend=True
    ), row=1, col=1)

# Panel 2: all-market daily total volume
daily_vol = (
    ohlcv_df.groupby(ohlcv_df['timestamp'].dt.date)['volume'].sum()
    .reset_index().rename(columns={'timestamp':'date'})
)
fig.add_trace(go.Bar(
    x=daily_vol['date'].astype(str), y=daily_vol['volume'],
    name='Total vol', marker_color='steelblue', showlegend=False,
    hovertemplate='Date: %{x}<br>Vol: $%{y:,.0f}<extra></extra>'
), row=2, col=1)

# Panel 3: volume-weighted spread (liquidity indicator)
mdf['wt_spread'] = mdf['spread'] * (mdf['volume'] + 1)
vwspread = mdf.groupby('timestamp').apply(
    lambda g: g['spread'].mean() if g['volume'].sum() < 1 else (g['wt_spread'].sum() / (g['volume'] + 1).sum())
).reset_index(name='vw_spread')
fig.add_trace(go.Scatter(
    x=vwspread['timestamp'], y=vwspread['vw_spread'],
    mode='lines', line=dict(color='darkorange', width=1), name='VW spread',
    hovertemplate='Time: %{x|%b %d %H:%M}<br>Avg spread: %{y:.4f}<extra></extra>', showlegend=False
), row=3, col=1)

fig.update_layout(
    title=f"<b>Trade Flow & Liquidity</b> — {MARKET}",
    barmode='stack', height=700, hovermode='x unified',
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9))
)
fig.update_yaxes(title_text='USDC vol', row=1, col=1)
fig.update_yaxes(title_text='USDC vol', row=2, col=1)
fig.update_yaxes(title_text='Spread', row=3, col=1)
fig.show()


In [ ]:

# ── Spread profile: liquidity vs probability level ────────────────────────
# Does the market price the high-prob bucket tighter (more liquid)?
spread_profile = (
    mdf.groupby('bucket_mid')
    .agg(avg_spread=('spread','mean'), avg_prob=('close','mean'),
         avg_vol=('volume','mean'), bucket=('bucket','first'))
    .reset_index().sort_values('bucket_mid')
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Spread vs Probability (each dot = one bucket)',
                                    'Volume vs Probability'])
for _, r in spread_profile.iterrows():
    hover = f"<b>Bucket {r['bucket']}</b><br>Avg prob: {r['avg_prob']:.1%}<br>Avg spread: {r['avg_spread']:.4f}<br>Avg vol/hr: ${r['avg_vol']:,.0f}"
    fig.add_trace(go.Scatter(
        x=[r['avg_prob']], y=[r['avg_spread']], mode='markers',
        marker=dict(size=8, color=prob_to_color(r['avg_prob']), line=dict(width=0.5, color='gray')),
        name=r['bucket'], showlegend=False,
        hovertemplate=hover + '<extra></extra>'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[r['avg_prob']], y=[r['avg_vol']+0.01], mode='markers',
        marker=dict(size=8, color=prob_to_color(r['avg_prob']), line=dict(width=0.5, color='gray')),
        name=r['bucket'], showlegend=False,
        hovertemplate=hover + '<extra></extra>'
    ), row=1, col=2)

fig.update_layout(title=f"<b>Liquidity Profile</b> — {MARKET}<br>"
                        "<sup>Hover over each dot to see the bucket. "
                        "Expect tighter spreads and higher volume near the peak probability bucket.</sup>",
                  height=420)
fig.update_xaxes(title_text='Avg probability', tickformat='.0%')
fig.update_yaxes(title_text='Avg bid-ask spread', row=1, col=1)
fig.update_yaxes(title_text='Avg hourly volume (USDC)', type='log', row=1, col=2)
fig.show()


In [ ]:

# ── Lead-lag: cross-correlation between bucket log-returns ─────────────────
# Computes corr(bucket_i @ t,  bucket_j @ t+lag) for lags -12h to +12h.
# Positive lag: column j leads row i (j moves first, i follows).

MAX_LAG = 12

# Build returns matrix (hours × buckets)
pm = price_matrix.copy()  # already built above, rows=timestamps, cols=bucket_mids
log_ret = np.log(pm / pm.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
bms = log_ret.columns.tolist()
R   = log_ret.values   # (T, N)
T, N = R.shape

# Compute cross-correlation matrix for each lag
def xcorr_matrix(R, lag):
    """Entry [i,j] = corr( R_i[t], R_j[t+lag] ) — positive lag means j leads i."""
    if lag == 0:
        return np.corrcoef(R.T)
    elif lag > 0:
        return np.corrcoef(R[:-lag].T, R[lag:].T)[:N, N:]
    else:
        k = -lag
        return np.corrcoef(R[k:].T, R[:-k].T)[:N, N:]

# Collect max-correlation lag for each pair
max_corr_lag = np.zeros((N, N))
max_corr_val = np.full((N, N), -np.inf)
lags = list(range(-MAX_LAG, MAX_LAG + 1))
xcorrs = {}
for lag in lags:
    C = xcorr_matrix(R, lag)
    xcorrs[lag] = C
    mask = C > max_corr_val
    max_corr_val = np.where(mask, C, max_corr_val)
    max_corr_lag = np.where(mask, lag, max_corr_lag)

# Label every ~5th bucket on axes; all readable on hover via custom data
step2 = max(1, N // 10)
tick_vals = list(range(0, N, step2))
tick_text = [mdf[mdf['bucket_mid'] == bms[i]]['bucket'].iloc[0] if len(mdf[mdf['bucket_mid'] == bms[i]]) else str(bms[i])
             for i in tick_vals]

bucket_labels_all = [mdf[mdf['bucket_mid'] == bm]['bucket'].iloc[0] if len(mdf[mdf['bucket_mid'] == bm]) else str(bm)
                     for bm in bms]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Lag-0 correlation (contemporaneous)',
                                    'Lead-lag: which bucket moves first? (hours)'])

fig.add_trace(go.Heatmap(
    z=xcorrs[0], colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    x=bms, y=bms,
    colorbar=dict(x=0.45, title='ρ', len=0.9),
    customdata=[[f"Row: {bucket_labels_all[i]}<br>Col: {bucket_labels_all[j]}<br>ρ={xcorrs[0][i,j]:.3f}"
                 for j in range(N)] for i in range(N)],
    hovertemplate='%{customdata}<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=max_corr_lag, colorscale='PiYG', zmid=0,
    x=bms, y=bms,
    colorbar=dict(x=1.0, title='Lead hrs', len=0.9),
    customdata=[[f"Row: {bucket_labels_all[i]}<br>Col: {bucket_labels_all[j]}<br>"
                 f"Best lag: {max_corr_lag[i,j]:+.0f}h  (ρ={max_corr_val[i,j]:.3f})"
                 for j in range(N)] for i in range(N)],
    hovertemplate='%{customdata}<extra></extra>',
), row=1, col=2)

for col in [1, 2]:
    fig.update_xaxes(tickvals=tick_vals, ticktext=tick_text, tickangle=45, tickfont=dict(size=8), row=1, col=col)
    fig.update_yaxes(tickvals=tick_vals, ticktext=tick_text, tickfont=dict(size=8), row=1, col=col)

fig.update_layout(
    title=f"<b>Cross-Correlations</b> — {MARKET}<br>"
          "<sup>Right panel: green = column leads row; pink = row leads column. Hover for exact lag & bucket.</sup>",
    height=520
)
fig.show()


In [ ]:

# ── Lead-lag by distance from modal probability ────────────────────────────
# Hypothesis: the high-probability (modal) bucket receives information first.
# Test: do high-prob bucket returns predict low-prob bucket returns at lag k>0?
#
# Group by average probability level — works regardless of bucket count.
# Boundaries auto-scale to the data so each group has ≥1 member.

avg_prob_each = pm.mean()   # average probability per bucket over the full period

# Choose thresholds by quantile so all groups are always populated
p75 = avg_prob_each.quantile(0.75)
p50 = avg_prob_each.quantile(0.50)

def prob_to_group(p):
    if p >= p75: return f'high (≥{p75:.1%} avg prob)'
    if p >= p50: return f'mid  ({p50:.1%}–{p75:.1%} avg prob)'
    return              f'low  (<{p50:.1%} avg prob)'

group_of = avg_prob_each.map(prob_to_group)
group_names_actual = avg_prob_each.sort_values(ascending=False).map(prob_to_group).unique().tolist()

print("Bucket group sizes:")
print(group_of.value_counts().sort_index().to_string())
high_bkts = [bucket_labels_all[i] for i, bm in enumerate(bms) if group_of[bm].startswith('high')]
print(f"\nHigh-prob buckets: {high_bkts[:5]}{'...' if len(high_bkts)>5 else ''}")

group_colors_dyn = {}
palette = ['crimson', 'darkorange', 'steelblue']
for name, col in zip(sorted(group_names_actual), palette):
    group_colors_dyn[name] = col

# Group-average log-returns
group_ret = {}
for g in group_names_actual:
    cols = [bm for bm in bms if group_of.get(bm) == g]
    if cols:
        group_ret[g] = log_ret[cols].mean(axis=1)

if len(group_ret) < 2:
    print("Only one group found — adjust quantile thresholds above.")
else:
    # Cross-correlate each ordered pair at each lag
    lag_range = list(range(-MAX_LAG, MAX_LAG + 1))
    xcorr_groups = {}
    for g1 in group_names_actual:
        for g2 in group_names_actual:
            if g1 not in group_ret or g2 not in group_ret: continue
            r1, r2 = group_ret[g1].values, group_ret[g2].values
            ccs = []
            for lag in lag_range:
                if lag == 0:
                    c = np.corrcoef(r1, r2)[0, 1]
                elif lag > 0:
                    valid = np.isfinite(r1[:-lag]) & np.isfinite(r2[lag:])
                    c = np.corrcoef(r1[:-lag][valid], r2[lag:][valid])[0,1] if valid.sum()>5 else np.nan
                else:
                    k = -lag
                    valid = np.isfinite(r1[k:]) & np.isfinite(r2[:-k])
                    c = np.corrcoef(r1[k:][valid], r2[:-k][valid])[0,1] if valid.sum()>5 else np.nan
                ccs.append(c)
            xcorr_groups[(g1, g2)] = ccs

    # Only plot off-diagonal pairs (g1 ≠ g2)
    pairs = [(g1, g2) for g1 in group_names_actual for g2 in group_names_actual
             if g1 != g2 and (g1, g2) in xcorr_groups]

    fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 4), sharey=True)
    if len(pairs) == 1: axes = [axes]

    for ax, (g1, g2) in zip(axes, pairs):
        ccs  = xcorr_groups[(g1, g2)]
        col  = group_colors_dyn.get(g2, 'steelblue')
        ax.bar(lag_range, ccs, color=[col if c >= 0 else 'lightgray' for c in ccs],
               alpha=0.85, width=0.7)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5, linestyle=':')
        peak_lag = lag_range[np.nanargmax(ccs)]
        ax.axvline(peak_lag, color='red', linewidth=1.2, linestyle='--',
                   label=f'peak lag={peak_lag:+d}h')
        short1 = g1.split('(')[0].strip()
        short2 = g2.split('(')[0].strip()
        ax.set_title(f'corr( {short1}[t],  {short2}[t+lag] )\n'
                     f'+ lag = {short2} leads {short1}', fontsize=8)
        ax.set_xlabel('Lag (hours)')
        ax.set_ylabel('Correlation' if ax is axes[0] else '')
        ax.legend(fontsize=8)
        ax.set_xlim(-MAX_LAG, MAX_LAG)

    plt.suptitle(
        f'Lead-Lag by Probability Level — {MARKET}\n'
        'Does the high-probability (modal) bucket move BEFORE low-probability (tail) buckets?\n'
        'Peak right of 0 → column group leads row group.',
        fontsize=9
    )
    plt.tight_layout()
    plt.show()


In [ ]:

# ── Time-to-expiry (DTE) effects ───────────────────────────────────────────
# As a market approaches resolution, does liquidity improve? Does vol spike?
# Expected: vol ↑ near expiry (event risk), spread ↓ (info revealed),
#           volume ↑ (positioning / resolution play).

dte_df = mdf[mdf['dte'].notna() & (mdf['dte'] >= 0) & (mdf['dte'] <= 120)].copy()
dte_df['dte_bin'] = pd.cut(dte_df['dte'], bins=list(range(0, 121, 5)), right=False)

# Return volatility per (bucket, DTE bin)
ret_df = (
    mdf.sort_values(['bucket_mid','timestamp'])
    .assign(log_ret=lambda df: np.log(df.groupby('bucket_mid')['close'].transform(lambda x: x / x.shift(1))))
)
ret_df['dte_bin'] = pd.cut(ret_df['dte'], bins=list(range(0, 121, 5)), right=False)

dte_stats = (
    ret_df[ret_df['dte'].notna() & (ret_df['dte'] >= 0) & (ret_df['dte'] <= 120)]
    .groupby('dte_bin', observed=True)
    .agg(vol=('log_ret', lambda x: x.std() * np.sqrt(24)),   # annualised to daily vol
         spread=('spread', 'mean'),
         volume=('volume', 'sum'),
         n=('log_ret', 'count'))
    .reset_index()
)
dte_stats['dte_mid'] = dte_stats['dte_bin'].apply(lambda x: x.mid if hasattr(x,'mid') else np.nan)
dte_stats = dte_stats.dropna(subset=['dte_mid']).sort_values('dte_mid', ascending=False)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=['Return Volatility (daily, annualised)',
                                    'Avg Bid-Ask Spread',
                                    'Total USDC Volume'])

fig.add_trace(go.Scatter(x=dte_stats['dte_mid'], y=dte_stats['vol'],
    mode='lines+markers', line=dict(color='crimson', width=1.5),
    hovertemplate='DTE: %{x:.0f} days<br>Vol: %{y:.4f}<extra></extra>', showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=dte_stats['dte_mid'], y=dte_stats['spread'],
    mode='lines+markers', line=dict(color='steelblue', width=1.5),
    hovertemplate='DTE: %{x:.0f} days<br>Spread: %{y:.4f}<extra></extra>', showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=dte_stats['dte_mid'], y=dte_stats['volume'],
    marker_color='seagreen', opacity=0.7,
    hovertemplate='DTE: %{x:.0f} days<br>Vol: $%{y:,.0f}<extra></extra>', showlegend=False), row=3, col=1)

fig.update_layout(
    title=f"<b>Time-to-Expiry Effects</b> — {MARKET}<br>"
          "<sup>x-axis = days to expiry (right=far, left=near resolution). "
          "Watch for vol spike and volume surge as DTE → 0.</sup>",
    height=600
)
fig.update_xaxes(title_text='Days to expiry', row=3, col=1, autorange='reversed')
fig.update_yaxes(title_text='Daily vol', row=1, col=1)
fig.update_yaxes(title_text='Spread', row=2, col=1)
fig.update_yaxes(title_text='Volume ($)', row=3, col=1)
fig.show()


In [ ]:

# ── Microstructure: autocorrelation + time-of-day patterns ────────────────
from statsmodels.graphics.tsaplots import plot_acf

# Use the core bucket's return series for ACF
core_bm  = [bm for bm in bms if group_of.get(bm) == 'core (rank 1–3, ~modal)']
core_ret = log_ret[core_bm].mean(axis=1).dropna() if core_bm else log_ret.iloc[:, 0].dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel 1: ACF of core bucket returns
plot_acf(core_ret, lags=48, alpha=0.05, ax=axes[0, 0])
axes[0, 0].set_title('ACF — Core bucket hourly log-returns\n(lags 0–48h)', fontsize=9)
axes[0, 0].set_xlabel('Lag (hours)')

# Panel 2: Hour-of-day patterns (UTC)
mdf['hour_utc'] = mdf['timestamp'].dt.hour
tod_vol    = mdf.groupby('hour_utc')['volume'].mean()
tod_spread = mdf.groupby('hour_utc')['spread'].mean()

ax2b = axes[0, 1].twinx()
axes[0, 1].bar(tod_vol.index, tod_vol.values, color='steelblue', alpha=0.6, label='Avg vol/hr')
ax2b.plot(tod_spread.index, tod_spread.values, color='crimson', linewidth=1.5, marker='o',
          markersize=3, label='Avg spread')
axes[0, 1].set_xlabel('Hour of day (UTC)')
axes[0, 1].set_ylabel('Avg volume (USDC)', color='steelblue')
ax2b.set_ylabel('Avg spread', color='crimson')
axes[0, 1].set_title('Time-of-day: volume and spread\n(when is the market most active & liquid?)', fontsize=9)
lines1, labs1 = axes[0, 1].get_legend_handles_labels()
lines2, labs2 = ax2b.get_legend_handles_labels()
axes[0, 1].legend(lines1+lines2, labs1+labs2, fontsize=8)

# Panel 3: Return distribution per group (violin / KDE)
for g, color in group_colors.items():
    if g not in group_ret: continue
    r = group_ret[g].replace([np.inf, -np.inf], np.nan).dropna()
    r_clip = r.clip(r.quantile(0.02), r.quantile(0.98))
    axes[1, 0].hist(r_clip, bins=50, density=True, alpha=0.5, color=color, label=g[:6])
    kde_x = np.linspace(r_clip.min(), r_clip.max(), 200)
    kde   = stats.gaussian_kde(r_clip)(kde_x)
    axes[1, 0].plot(kde_x, kde, color=color, linewidth=1.5)
axes[1, 0].set_title('Return distribution by bucket group\n(clipped to 2nd-98th pctile)', fontsize=9)
axes[1, 0].set_xlabel('Hourly log-return')
axes[1, 0].legend(fontsize=8)
axes[1, 0].axvline(0, color='black', linewidth=0.5, linestyle=':')

# Panel 4: Probability distribution of the core bucket at different DTEs
dte_groups = [(90, 120, 'far (90–120d)'), (30, 60, 'mid (30–60d)'),
              (5, 20, 'near (5–20d)'), (0, 5, 'final (0–5d)')]
for lo, hi, label in dte_groups:
    sub = mdf[(mdf['dte'] >= lo) & (mdf['dte'] < hi)]
    if sub.empty: continue
    probs = sub['close'].values
    axes[1, 1].hist(probs, bins=30, density=True, alpha=0.5, label=label, range=(0,1))
axes[1, 1].set_title('All-bucket probability distribution by DTE\n(does distribution sharpen near expiry?)', fontsize=9)
axes[1, 1].set_xlabel('Bucket probability (0–1)')
axes[1, 1].legend(fontsize=8)

plt.suptitle(f'Microstructure — {MARKET}', fontsize=11)
plt.tight_layout()
plt.show()
